<a href="https://colab.research.google.com/github/erinmcfadden3/Lab6-SearchAlgorithms/blob/main/final_exam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CSC 2053 — Platform Based Computing
## Final Exam, Part 2: North Carolina Radio Deep Dive

**Spring 2026**

---

### Before you begin — complete all three steps below in order.

> **This is an open-resource exam.** You may use any resource  —
> documentation, notes, the internet, or any other tool you find helpful.
> The only requirement: You may **not** share answers with other students or copy another student's work.
>
> **Citation requirement:** You don't have to list each source for every question, but please include any source you've used in the exam as a whole in the citation cell.

---

**This is Part 2 of the Final Exam — worth 50 points.**

**Point values:** Q1–Q5: 2 pts each · Q6–Q11: 3 pts each · Q12–Q14: 6 pts each · Q15: 4 pts · **Part 2 Total: 50 pts**

There are **many ways** to answer a data-related question in Python — you won't be judged on your approach, only your result. For each question, make sure your answer is visible: the last line of a cell displays automatically in Colab, or use display() for tables.

## Step 1 — Save Your Work

Colab does **not** reliably autosave. Protect yourself:

- **Ctrl+S** (or File → Save) saves to your current session.
- **File → Download → Download .ipynb** saves a local copy to your computer.

You will hit two save checkpoints mid-exam as reminders. Do not skip them.

## Step 2 — Imports

In [1]:
import numpy as np
import pandas as pd
import math
from IPython.display import display

print("Libraries loaded.")

Libraries loaded.


## Step 3 — Load North Carolina Data

Run all three cells below before answering any questions.

In [2]:
# Load NC radio stations
url = "https://raw.githubusercontent.com/CSC-2053-Spring-26-100/lab10-python-through-state-radio/main/states/NC_stations.csv"
df = pd.read_csv(url)
print(f"Loaded {len(df):,} stations for NC")
df.head()

Loaded 836 stations for NC


,band,callsign,frequency,city,state,county,lat,lon,class,format,...,power,haat,market,rank,population,nfl,nba,nhl,mlb,facility_id
0,FM,W217CO,91.3,WILSON,NC,WILSON,35.747861,78.020139,D,CONTEMPORARY CHRISTIAN,...,0.055,55.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148153.0
1,FM,W202CM,88.3,"CULLOWHEE, ETC.",NC,JACKSON,35.313944,83.201139,D,NEWS/TALK,...,0.019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71879.0
2,FM,W206CL,89.1,"CLYDE, ETC.",NC,HAYWOOD,35.568417,82.907333,D,NEWS/TALK,...,0.010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71878.0
3,FM,W211CI,90.1,ROCKINGHAM,NC,RICHMOND,34.961500,79.841167,D,RELIGIOUS TEACHING,...,0.024,69.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,106479.0
4,FM,W248DN,97.5,CASHIERS,NC,JACKSON,35.134722,83.099111,D,EASY LISTENING,...,0.150,538.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,142647.0


In [3]:
# Clean: remove non-conforming D-callsign rows
bad = df[df['callsign'].str.startswith('D', na=False)]
print(f"Removing {len(bad)} non-conforming rows")
df = df[~df['callsign'].str.startswith('D', na=False)].reset_index(drop=True)
print(f"Stations after cleaning: {len(df):,}")

Removing 0 non-conforming rows
Stations after cleaning: 836


In [4]:
# Fix longitude sign (stored as positive in source data — US lons should be negative)
df['lon'] = df['lon'] * -1

# Split into AM and FM DataFrames
am_df = df[df['band'] == 'AM'].copy().reset_index(drop=True)
fm_df = df[df['band'] == 'FM'].copy().reset_index(drop=True)
print(f"AM: {len(am_df)} stations | FM: {len(fm_df)} stations")

AM: 202 stations | FM: 634 stations


## Citations

**List any external source you used below.**

*(Double-click this cell to edit.)*

-

**Your name:** *Erin McFadden*

---
## Section 1 — First Look at the Landscape

### Question 1 — Station Count by Band (2 pts)
**Concepts:** `len()`, `.value_counts()`

How many total radio stations are in North Carolina's dataset?
Print the total, then display a breakdown by band (AM vs FM) using `.value_counts()`.

In [8]:
# Q1 — YOUR CODE HERE
total = len(df)
print(f"Total radio stations in NC: {total}")

band_breakdown = df['band'].value_counts()
print("\nBreakdown by band:")
print(band_breakdown)


Total radio stations in NC: 836

Breakdown by band:
band
FM    634
AM    202
Name: count, dtype: int64


### Question 2 — First Look at the Data (2 pts)
**Concepts:** `.head()`, `.dtypes`

**a)** Display the first 5 rows of `df`.

**b)** Print all column names and their data types using `.dtypes`.

In [9]:
# Q2a — Display the first 5 rows
df.head()

,band,callsign,frequency,city,state,county,lat,lon,class,format,...,power,haat,market,rank,population,nfl,nba,nhl,mlb,facility_id
0,FM,W217CO,91.3,WILSON,NC,WILSON,35.747861,-78.020139,D,CONTEMPORARY CHRISTIAN,...,0.055,55.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,148153.0
1,FM,W202CM,88.3,"CULLOWHEE, ETC.",NC,JACKSON,35.313944,-83.201139,D,NEWS/TALK,...,0.019,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71879.0
2,FM,W206CL,89.1,"CLYDE, ETC.",NC,HAYWOOD,35.568417,-82.907333,D,NEWS/TALK,...,0.010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,71878.0
3,FM,W211CI,90.1,ROCKINGHAM,NC,RICHMOND,34.961500,-79.841167,D,RELIGIOUS TEACHING,...,0.024,69.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,106479.0
4,FM,W248DN,97.5,CASHIERS,NC,JACKSON,35.134722,-83.099111,D,EASY LISTENING,...,0.150,538.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,142647.0


In [10]:
# Q2b — Print column names and data types
print(df.dtypes)

band            object
callsign        object
frequency      float64
city            object
state           object
county          object
lat            float64
lon            float64
class           object
format          object
slogan          object
owner           object
power          float64
haat           float64
market          object
rank           float64
population      object
nfl             object
nba             object
nhl             object
mlb             object
facility_id    float64
dtype: object


### Question 3 — Cities and Counties (2 pts)
**Concepts:** `.nunique()`, `.value_counts()`

Using `df` (all stations):

**a)** How many unique cities have radio stations in NC? How many unique counties?

**b)** Which city has the most radio stations? Display the top 5 cities by station count.

In [11]:
# Q3a — Unique cities and counties
unique_cities = df['city'].nunique()
print(f"Unique cities with radio stations in NC: {unique_cities}")

Unique cities with radio stations in NC: 274


In [12]:
# Q3b — Top 5 cities by station count
unique_counties = df['county'].nunique()
print(f"Unique counties with radio stations in NC: {unique_counties}")

Unique counties with radio stations in NC: 100


### Question 4 — Longest City Name (2 pts)
**Concepts:** `.str.len()`, `.sort_values()`, `.iloc[]`

Which city name in the NC dataset is the longest by character count? Note, you may find that two cities are tied with the longest amount of characters in their name.

In [18]:
# Q4 — YOUR CODE HERE
df['length'] = df['city'].str.len()
df_sorted = df.sort_values(by='length', ascending=False)

longest_name = df_sorted.iloc[0]['city']
max_length = df_sorted.iloc[0]['length']

print(f"Longest city name: {longest_name}")
print(f"\nNumber of characters: {max_length}")

print("\nTop results (checking for ties):")
print(df_sorted[['city', 'length']].head())

Longest city name: LOCKWOODS FOLLY TOWN

Number of characters: 20

Top results (checking for ties):
                     city  length
405  LOCKWOODS FOLLY TOWN      20
454  BOILING SPRING LAKES      20
425    WRIGHTSVILLE BEACH      18
592     EAST FAYETTEVILLE      17
760     BURLINGTON-GRAHAM      17


### Question 5 — Format Dominance by Market (2 pts)
**Concepts:** `.groupby()`, `.value_counts()`, `.apply()`

For each of the **top 5 markets** by total station count, find the most common
programming format.

Display a table with columns `market`, `top_format`, and `count`.

*Hint: group by market, then within each group find the mode of the format column.*

In [22]:
# Q5 — YOUR CODE HERE
top_market_names = df['market'].value_counts().head(5).index

df_top_markets = df[df['market'].isin(top_market_names)]
result = df_top_markets.groupby('market')['format'].apply(lambda x: x.value_counts().head(1))

result_df = result.reset_index()
result_df.columns = ['market', 'top_format', 'count']

print(result_df)



                                market          top_format  count
0                            ASHEVILLE  RELIGIOUS TEACHING      7
1         CHARLOTTE-GASTONIA-ROCK HILL  RELIGIOUS TEACHING     16
2  GREENSBORO-WINSTON SALEM-HIGH POINT  RELIGIOUS TEACHING     19
3     GREENVILLE-NEW BERN-JACKSONVILLE              SPORTS     10
4                       RALEIGH-DURHAM             COUNTRY     10


---
## Section 2 — Signal Strength & Geography

### Question 6 — Top 10 by HAAT (3 pts)
**Concepts:** `.dropna()`, `.nlargest()` or `.sort_values()`

**HAAT** (Height Above Average Terrain) is a key measure of a station's geographic reach.

Using `fm_df`, display the **top 10 FM stations by HAAT**, sorted descending.
Show `callsign`, `city`, `county`, `haat`, and `power`.

*Note: HAAT is only tracked for FM stations in this dataset.*

In [23]:
# Q6 — YOUR CODE HERE
fm_df = df[df['band'] == 'FM'].dropna(subset = ['haat'])

top_haat = fm_df[['callsign', 'city', 'county', 'haat', 'power']].sort_values(by='haat', ascending=False).head(10)

print(f"Top 10 FM stations by HAAT:")
print(top_haat)

Top 10 FM stations by HAAT:
    callsign            city    county   haat    power
479     WMIT  BLACK MOUNTAIN  BUNCOMBE  942.0   36.000
495     WNCW        SPINDALE  BUNCOMBE  926.0   17.000
451     WKSF        OLD FORT   HAYWOOD  786.0   53.000
399     WFQS        FRANKLIN     MACON  702.0    0.265
492     WNCC        FRANKLIN   JACKSON  681.0    0.115
568     WTPT     FOREST CITY      POLK  619.0   93.000
368  WCVP-FM    ROBBINSVILLE  CHEROKEE  612.0    0.120
528     WRAL         RALEIGH  JOHNSTON  606.0   98.000
531     WRDU     WAKE FOREST      NASH  600.0  100.000
595     WWMY  BEECH MOUNTAIN     AVERY  597.0    0.150


### Question 7 — Counties by Station Count (3 pts)
**Concepts:** `.groupby()`, `.size()`, `.sort_values()`

Using `df` (all stations), rank all NC counties by total station count, descending.
Display the full ranked list.

Then answer: how many counties have **fewer than 5 stations**?

In [28]:
# Q7a — Ranked list of counties by station count
county_count = df.groupby('county').size()
ranked = county_count.sort_values(ascending=False)
print("Ranking of all NC counties by station count (descending):")
print(ranked)

Ranking of all NC counties by station count (descending):
county
WAKE           39
BUNCOMBE       38
GUILFORD       37
MECKLENBURG    25
FORSYTH        24
               ..
HORRY           1
GREENE          1
GREENVILLE      1
RABUN           1
WASHINGTON      1
Length: 100, dtype: int64


In [30]:
# Q7b — How many counties have fewer than 5 stations?
below_5 = ranked[ranked < 5]
print(f"Number of counties with fewer than 5 stations: {len(below_5)}")

Number of counties with fewer than 5 stations: 42


### Question 8 — Format Diversity by County (3 pts)
**Concepts:** `.groupby()`, `.nunique()`

Which county offers listeners the **most diverse programming**?
Using `df`, calculate the number of unique formats per county and display the top 10,
sorted descending.

*Hint: `.groupby('county')['format'].nunique()`*

In [31]:
# Q8 — YOUR CODE HERE
county_diverse = df.groupby('county')['format'].nunique()
top_diverse = county_diverse.sort_values(ascending=False).head(10)
print("Top 10 most diverse counties (descending):")
print(top_diverse)

Top 10 most diverse counties (descending):
county
WAKE          19
BUNCOMBE      18
GUILFORD      18
CUMBERLAND    15
DURHAM        15
NASH          14
GASTON        14
FORSYTH       14
ONSLOW        13
CRAVEN        12
Name: format, dtype: int64


> **Save checkpoint (1/2):** File → Download → Download .ipynb and keep it somewhere safe.
> Do this now before continuing.

### Question 9 — Who Controls Asheville's Airwaves? (3 pts)
**Concepts:** boolean filtering, `.value_counts()`

Asheville sits in **Buncombe County** — one of North Carolina's most culturally distinct communities, and a market Chairman Carr might find interesting.

**a)** Filter `df` to only stations in Buncombe County. How many stations serve the area?

**b)** Display a breakdown of ownership using `.value_counts()`. Who is the top owner, and how many stations do they control?

In [ ]:
# Q9a — Filter to Buncombe County and count


In [ ]:
# Q9b — Ownership breakdown


---
## Section 3 — Frequencies & Ownership

### Question 10 — Most Common Frequencies (3 pts)
**Concepts:** `.value_counts()`, band filtering

**a)** What is the most common FM frequency in NC? How many stations share it?

**b)** What is the most common AM frequency in NC? How many stations share it?

Display the top 5 for each band.

In [ ]:
# Q10a — FM: YOUR CODE HERE


In [ ]:
# Q10b — AM: YOUR CODE HERE


### Question 11 — Ownership Concentration (3 pts)
**Concepts:** `.value_counts()`, arithmetic, f-strings

**a)** Who is the top owner in NC by station count?

**b)** What **percentage** of all NC stations does the top owner control?
Round to 1 decimal place.

**c)** Display the top 10 owners by station count.

In [ ]:
# Q11a — Top owner and percentage


In [ ]:
# Q11b — Top 10 owners


> **Save checkpoint (2/2):** File → Download → Download .ipynb again.
> Four questions left — you've got this.

---
## Section 4 — Sports on the Dial

### Question 12 — Sports Coverage Summary (6 pts)
**Concepts:** boolean filtering, `pd.DataFrame()`

 Build a summary table showing how many NC stations carry each major sports league. Your table should have columns Sport and Station Count, with rows for NFL, MLB, NBA, and NHL — sorted descending by count.                                                                                                                                                                                                    
                                                            
  Note: The nfl, mlb, nba, and nhl columns each contain the name of the affiliated franchise, or are null — notna() — if the station carries no affiliation for that league.

Your table should have columns `Sport` and `Station Count`, with rows for
NFL, MLB, NBA, and NHL — sorted descending by count.

In [ ]:
# Q12 — YOUR CODE HERE


### Question 13 — Panthers West, Hurricanes East? (6 pts)
**Concepts:** NumPy boolean masking, geographic filtering

Raleigh sits at approximately **-79.0° longitude** — the state capital, home of the Carolina Hurricanes, and a natural east-west dividing line through North Carolina.

**a)** Filter `df` to only Panthers NFL affiliates, then use a NumPy boolean mask to count how many fall **west** of -79.0° and how many fall **east**. Print both counts.

**b)** Do the same for Hurricanes NHL affiliates — how many fall west vs. east of -79.0°?

*Hint: west = `lon < -79.0`, east = `lon >= -79.0`*

In [ ]:
# Q13a — Panthers east vs. west of -79.0°


In [ ]:
# Q13b — Hurricanes east vs. west of -79.0°


### Question 14 — How Close Are Panthers Stations to Charlotte? (6 pts)
**Concepts:** boolean filtering, `.apply()`, sorting

The Carolina Panthers play at **Bank of America Stadium** in Charlotte.
Coordinates: **35.2258° N, 80.8528° W**. A Haversine function is provided in the cell below — run it first.

**a)** Using boolean filtering, create a DataFrame `panthers_df` containing only stations
where the `nfl` column equals `'Panthers'`.

**b)** Add a column `miles_to_stadium` to `panthers_df` by applying the Haversine function
to each station's `lat` and `lon` against the stadium coordinates above.

**c)** Sort by `miles_to_stadium` ascending and display `callsign`, `city`, `county`,
and `miles_to_stadium`.

In [ ]:
# Provided — run this cell as-is
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

print("haversine() ready")

In [ ]:
# Q14a — Create panthers_df


In [ ]:
# Q14b — Add miles_to_stadium column


In [ ]:
# Q14c — Sort and display


---
## Section 5 — Your Briefing for Chairman Carr

### Question 15 — Written Reflection (4 pts)

In **3–5 sentences**, summarize what the North Carolina radio data tells us.

Address at least two of the following:
- Ownership concentration — who controls the dial?
- Geographic coverage — where are the gaps?
- Sports radio — which teams and leagues are well-served? Which are not?
- Signal strength — what patterns did you notice in power or HAAT?

What is the single most surprising finding you'd highlight for Chairman Carr?

*(Double-click this cell to type your answer.)*

> **Your answer:**

---
## Submission

1. Make sure all cells have been run and your written responses are filled in
2. Confirm your Citations cell is complete
3. **File → Download → Download .ipynb**
4. Upload to your Final Exam GitHub Classroom repository
5. That is all. You do not need to submit the file on Blackboard.